# REV1-04 — Figure 7: Clean Representative Grid Selection

Notebook ini memperbaiki pemilihan grid representatif untuk **empat tahun dengan maximum simultaneous drought-core extent nasional tertinggi**.

Prinsipnya:

1. empat tahun tetap ditentukan **dinamis** dari ranking M2A final;
2. pentad puncak nasional tetap sama;
3. kandidat tetap dibatasi pada **largest connected P20 drought-core cluster** pada pentad puncak;
4. setiap kandidat harus mempunyai event M2A yang benar-benar **mencakup pentad puncak**;
5. dari kandidat tersebut dipilih lintasan dengan **clean lifecycle score** tertinggi.

`Clean lifecycle score` memprioritaskan:

- onset yang turun secara konsisten;
- kondisi pra-onset yang tidak sudah berada di drought core;
- drought core yang dalam dan relatif stabil;
- recovery yang jelas;
- **tidak langsung jatuh lagi ke <P20** selama beberapa pentad setelah recovery;
- **tidak ada event start baru** segera setelah recovery.

Notebook juga menyimpan **seluruh kandidat dan skornya**, sehingga pemilihan grid dapat diaudit dan direproduksi.

> Catatan metodologis: pemilihan berdasarkan kejernihan lintasan adalah *illustrative selection criterion*. Jika gambar digunakan dalam skripsi/paper, kriteria pemilihan sebaiknya disebutkan secara singkat pada caption/metode.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from scipy import ndimage

warnings.filterwarnings("once")

# ============================================================
# PATH
# ============================================================
BASE = Path(r"D:\ERA5_LAND")

DET_DIR = BASE / "output_fd_03b_sensitivity"
CLIM_DIR = BASE / "output_fd_03d_paper_climatology"

DETECTION_FILE = DET_DIR / "fd_detection_M2A_1995_2025_indonesia.nc"
PERCENTILE_FILE = BASE / "output_rzsm" / "rzsm_percentile_gringorten_1995_2025.nc"
ANNUAL_CSV = CLIM_DIR / "annual_spatial_metrics_1996_2024.csv"

OUTPUT_DIR = BASE / "output_fd_REV1_04_fig7_clean"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# CLEAN-LIFECYCLE SETTINGS
# ============================================================
PRE_ONSET_PENTADS = 4
POST_RECOVERY_PENTADS = 4
TOP_N_PREVIEW = 3

# Hard preference. Jika tidak ada kandidat strict-pass,
# notebook tetap memilih skor tertinggi tetapi memberi flag.
STRICT_ONSET_NONINCREASE_FRAC = 0.80
STRICT_MAX_MINIMUM_PERCENTILE = 10.0

FIG_DPI = 300

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "font.size": 10.5,
    "axes.titleweight": "bold",
})

def safe_savefig(fig, filename):
    path = OUTPUT_DIR / filename
    fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
    print("Saved:", path)


## 1. Preflight dan load data final

In [ ]:
required = [DETECTION_FILE, PERCENTILE_FILE, ANNUAL_CSV]
missing = [p for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "File wajib belum tersedia:\n" + "\n".join(str(p) for p in missing)
    )

annual = pd.read_csv(ANNUAL_CSV)

det = xr.open_dataset(
    DETECTION_FILE,
    chunks={"time_index": 73},
    mask_and_scale=False,
)

valid = (det["valid_grid_mask"] == 1).compute()

# Percentile -> detection time ordering
pct_ds = xr.open_dataset(PERCENTILE_FILE)
pct4 = pct_ds["rzsm_percentile"].transpose(
    "year", "pentad", "latitude", "longitude"
)

pct_time = (
    pct4.stack(time_index=("year", "pentad"))
    .transpose("time_index", "latitude", "longitude")
    .reset_index("time_index", drop=True)
    .assign_coords(
        time_index=np.arange(
            pct4.sizes["year"] * pct4.sizes["pentad"]
        )
    )
)

if pct_time.sizes["time_index"] != det.sizes["time_index"]:
    raise ValueError("Percentile and detection time axes differ.")

lat = np.asarray(det["latitude"].values, dtype=float)
lon = np.asarray(det["longitude"].values, dtype=float)
coslat = np.cos(np.deg2rad(lat))

full_year = np.asarray(det["year"].values, dtype=int)
full_pentad = np.asarray(det["pentad"].values, dtype=int)

labels_full = [
    f"{yy}-P{pp:02d}"
    for yy, pp in zip(full_year, full_pentad)
]

top_years = (
    annual.nlargest(4, "maximum_drought_core_area_percent")
    [["year", "maximum_drought_core_area_percent", "pentad_of_maximum_core_area"]]
    .reset_index(drop=True)
)

print("Empat tahun drought-core extent terbesar:")
display(top_years)

expected = [2003, 1997, 2015, 2014]
observed = top_years["year"].astype(int).tolist()
if observed != expected:
    print(
        "CATATAN: ranking dinamis berbeda dari audited run sebelumnya.\n"
        f"Audited={expected}, current={observed}"
    )


## 2. Fungsi cluster dan kandidat

Kandidat hanya berasal dari **largest connected drought-core cluster** pada pentad puncak nasional. Jadi tahun dan episode nasional tetap sama dengan Figure 7 lama; yang berubah hanya cara memilih satu grid di dalam cluster utama.


In [ ]:
def get_peak_time_index(year, pentad):
    match = np.flatnonzero(
        (full_year == int(year))
        & (full_pentad == int(pentad))
    )
    if match.size != 1:
        raise ValueError(
            f"Expected one time index for {year} P{pentad}; found {match.size}."
        )
    return int(match[0])


def largest_core_cluster(year, pentad):
    peak_t = get_peak_time_index(year, pentad)

    core = (
        det["fd_drought_core"]
        .isel(time_index=peak_t)
        .values == 1
    )
    core = core & valid.values

    structure = np.ones((3, 3), dtype=np.int8)  # 8-neighbour
    labelled, nlab = ndimage.label(core, structure=structure)

    if nlab == 0:
        raise ValueError(f"No drought-core grid at {year} P{pentad}.")

    best_label = None
    best_weight = -np.inf

    for lab_id in range(1, nlab + 1):
        iy, ix = np.where(labelled == lab_id)
        if iy.size == 0:
            continue
        weight = float(np.sum(coslat[iy]))
        if weight > best_weight:
            best_weight = weight
            best_label = lab_id

    iy, ix = np.where(labelled == best_label)
    w = coslat[iy]

    centroid_lat = float(np.average(lat[iy], weights=w))
    centroid_lon = float(np.average(lon[ix], weights=w))

    return {
        "peak_time_index": peak_t,
        "iy": iy.astype(int),
        "ix": ix.astype(int),
        "centroid_lat": centroid_lat,
        "centroid_lon": centroid_lon,
        "cluster_grid_count": int(iy.size),
        "cluster_weight": best_weight,
    }


def covering_event_for_grid(peak_t, gy, gx):
    starts = np.flatnonzero(
        det["fd_event_start"]
        .isel(latitude=gy, longitude=gx)
        .values == 1
    )

    if starts.size == 0:
        return None

    durations = (
        det["duration_pentads_at_start"]
        .isel(latitude=gy, longitude=gx)
        .values
    )

    covering = [
        int(s)
        for s in starts
        if int(durations[s]) > 0
        and int(s) <= peak_t < int(s + durations[s])
    ]

    if not covering:
        return None

    # Jika ada lebih dari satu, ambil start terbaru sebelum peak.
    return max(covering)


## 3. Clean-lifecycle score

Skor bukan ukuran "keparahan". Skor hanya digunakan untuk mencari contoh lintasan yang mudah dibaca.

Bobot:

- 30 poin: tidak re-drop ke `<P20` dalam 4 pentad setelah recovery;
- 10 poin: tidak ada event start baru dalam 4 pentad setelah recovery;
- 15 poin: onset monoton/non-increasing;
- 10 poin: pra-onset tidak sudah berada di drought core;
- 10 poin: proporsi drought-core trajectory berada di `≤P10`;
- 10 poin: stabilitas drought core;
- 10 poin: post-recovery mean cukup tinggi;
- 5 poin: minimum event cukup dalam.

Jarak ke centroid **tidak menentukan skor**, tetapi digunakan sebagai *tie-breaker* agar kandidat yang sama-sama bersih tetap cenderung berada di bagian inti cluster.


In [ ]:
def _fraction(condition):
    arr = np.asarray(condition, dtype=bool)
    if arr.size == 0:
        return np.nan
    return float(np.mean(arr))


def candidate_metrics(year, peak_pentad, cluster, gy, gx, start_t):
    ntime = det.sizes["time_index"]
    peak_t = int(cluster["peak_time_index"])
    st = int(start_t)

    onset_time = int(
        det["onset_time_pentads_at_start"]
        .isel(time_index=st, latitude=gy, longitude=gx)
        .values
    )
    duration = int(
        det["duration_pentads_at_start"]
        .isel(time_index=st, latitude=gy, longitude=gx)
        .values
    )

    onset_speed = float(
        det["onset_speed_at_start"]
        .isel(time_index=st, latitude=gy, longitude=gx)
        .values
    )
    severity = float(
        det["severity_p40_at_start"]
        .isel(time_index=st, latitude=gy, longitude=gx)
        .values
    )
    minimum = float(
        det["minimum_percentile_at_start"]
        .isel(time_index=st, latitude=gy, longitude=gx)
        .values
    )

    crossing = st + onset_time
    termination = st + duration

    # Guardrail
    if not (0 <= st < crossing < ntime):
        return None
    if not (crossing <= peak_t < termination):
        return None
    if termination >= ntime:
        return None

    left_pre = max(0, st - PRE_ONSET_PENTADS)
    right_post = min(ntime, termination + POST_RECOVERY_PENTADS + 1)

    # Complete post-recovery window is preferred.
    post_complete = (
        termination + POST_RECOVERY_PENTADS < ntime
    )

    y_pre = np.asarray(
        pct_time.isel(
            time_index=slice(left_pre, st),
            latitude=gy,
            longitude=gx,
        ).values,
        dtype=float,
    )

    y_onset = np.asarray(
        pct_time.isel(
            time_index=slice(st, crossing + 1),
            latitude=gy,
            longitude=gx,
        ).values,
        dtype=float,
    )

    # Exclude termination because termination is recovery >=P20.
    y_core = np.asarray(
        pct_time.isel(
            time_index=slice(crossing, termination),
            latitude=gy,
            longitude=gx,
        ).values,
        dtype=float,
    )

    y_post = np.asarray(
        pct_time.isel(
            time_index=slice(termination + 1, right_post),
            latitude=gy,
            longitude=gx,
        ).values,
        dtype=float,
    )

    recovery_value = float(
        pct_time.isel(
            time_index=termination,
            latitude=gy,
            longitude=gx,
        ).values
    )

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------
    if len(y_onset) >= 2:
        onset_nonincrease_frac = _fraction(np.diff(y_onset) <= 0)
    else:
        onset_nonincrease_frac = np.nan

    pre_noncore_frac = _fraction(y_pre >= 20) if y_pre.size else np.nan
    core_low_frac = _fraction(y_core <= 10) if y_core.size else np.nan

    if y_core.size >= 2 and np.isfinite(y_core).all():
        core_std = float(np.std(y_core))
    else:
        core_std = 0.0 if y_core.size == 1 else np.nan

    # 1 = plateau sangat stabil; 0 = sangat noisy
    core_stability = (
        max(0.0, 1.0 - min(core_std / 15.0, 1.0))
        if np.isfinite(core_std)
        else 0.0
    )

    post_min = float(np.nanmin(y_post)) if y_post.size else np.nan
    post_mean = float(np.nanmean(y_post)) if y_post.size else np.nan

    immediate_relapse = (
        bool(np.any(y_post < 20))
        if y_post.size
        else True
    )

    post_start = termination + 1
    post_end = min(
        ntime,
        termination + POST_RECOVERY_PENTADS + 1,
    )

    second_event_start = bool(
        np.any(
            det["fd_event_start"]
            .isel(
                time_index=slice(post_start, post_end),
                latitude=gy,
                longitude=gx,
            )
            .values == 1
        )
    )

    # Recovery rise during the last up-to-3 transitions before termination.
    rec_left = max(crossing, termination - 3)
    y_rec = np.asarray(
        pct_time.isel(
            time_index=slice(rec_left, termination + 1),
            latitude=gy,
            longitude=gx,
        ).values,
        dtype=float,
    )
    recovery_positive_frac = (
        _fraction(np.diff(y_rec) >= 0)
        if y_rec.size >= 2
        else np.nan
    )

    # --------------------------------------------------------
    # Score components (0-100)
    # --------------------------------------------------------
    score_no_relapse = 30.0 if (post_complete and not immediate_relapse) else 0.0
    score_no_second = 10.0 if (post_complete and not second_event_start) else 0.0

    score_onset = 15.0 * (
        onset_nonincrease_frac
        if np.isfinite(onset_nonincrease_frac)
        else 0.0
    )

    score_pre = 10.0 * (
        pre_noncore_frac
        if np.isfinite(pre_noncore_frac)
        else 0.0
    )

    score_core_low = 10.0 * (
        core_low_frac
        if np.isfinite(core_low_frac)
        else 0.0
    )

    score_core_stability = 10.0 * core_stability

    # 20 percentile = baru recovery; 50 = recovery kuat.
    if np.isfinite(post_mean):
        post_strength = np.clip((post_mean - 20.0) / 30.0, 0.0, 1.0)
    else:
        post_strength = 0.0
    score_post = 10.0 * float(post_strength)

    min_depth = np.clip((20.0 - minimum) / 20.0, 0.0, 1.0)
    score_minimum = 5.0 * float(min_depth)

    clean_score = (
        score_no_relapse
        + score_no_second
        + score_onset
        + score_pre
        + score_core_low
        + score_core_stability
        + score_post
        + score_minimum
    )

    dist2 = (
        (lat[gy] - cluster["centroid_lat"]) ** 2
        + (
            (lon[gx] - cluster["centroid_lon"])
            * np.cos(np.deg2rad(cluster["centroid_lat"]))
        ) ** 2
    )
    centroid_distance_deg = float(np.sqrt(dist2))

    strict_pass = bool(
        post_complete
        and (not immediate_relapse)
        and (not second_event_start)
        and np.isfinite(onset_nonincrease_frac)
        and onset_nonincrease_frac >= STRICT_ONSET_NONINCREASE_FRAC
        and minimum <= STRICT_MAX_MINIMUM_PERCENTILE
    )

    return {
        "year": int(year),
        "national_peak_pentad": int(peak_pentad),
        "peak_time_index": peak_t,
        "start_time_index": st,
        "crossing_time_index": int(crossing),
        "termination_time_index": int(termination),
        "latitude_index": int(gy),
        "longitude_index": int(gx),
        "latitude": float(lat[gy]),
        "longitude": float(lon[gx]),
        "cluster_centroid_lat": float(cluster["centroid_lat"]),
        "cluster_centroid_lon": float(cluster["centroid_lon"]),
        "cluster_grid_count": int(cluster["cluster_grid_count"]),
        "centroid_distance_deg": centroid_distance_deg,
        "onset_time_pentads": onset_time,
        "onset_speed_pp_per_pentad": onset_speed,
        "duration_pentads": duration,
        "severity_p40": severity,
        "minimum_percentile": minimum,
        "recovery_value": recovery_value,
        "post_recovery_min": post_min,
        "post_recovery_mean": post_mean,
        "onset_nonincrease_fraction": onset_nonincrease_frac,
        "pre_noncore_fraction": pre_noncore_frac,
        "core_le_p10_fraction": core_low_frac,
        "core_std": core_std,
        "core_stability": core_stability,
        "recovery_positive_fraction": recovery_positive_frac,
        "post_window_complete": bool(post_complete),
        "immediate_relapse_below_p20": bool(immediate_relapse),
        "second_event_start_post_recovery": bool(second_event_start),
        "strict_clean_pass": strict_pass,
        "clean_score": float(clean_score),
    }


## 4. Scan seluruh kandidat pada empat tahun top

In [ ]:
all_rows = []

for row in top_years.itertuples(index=False):
    year = int(row.year)
    peak_pentad = int(row.pentad_of_maximum_core_area)

    cluster = largest_core_cluster(year, peak_pentad)
    peak_t = int(cluster["peak_time_index"])

    print(
        f"\n{year} P{peak_pentad:02d} | "
        f"largest cluster = {cluster['cluster_grid_count']:,} grid"
    )

    n_cover = 0

    for gy, gx in zip(cluster["iy"], cluster["ix"]):
        st = covering_event_for_grid(
            peak_t,
            int(gy),
            int(gx),
        )
        if st is None:
            continue

        metrics = candidate_metrics(
            year=year,
            peak_pentad=peak_pentad,
            cluster=cluster,
            gy=int(gy),
            gx=int(gx),
            start_t=int(st),
        )

        if metrics is not None:
            all_rows.append(metrics)
            n_cover += 1

    print("Covering-event candidates:", f"{n_cover:,}")

candidates = pd.DataFrame(all_rows)

if candidates.empty:
    raise RuntimeError("Tidak ada kandidat yang ditemukan.")

# Ranking:
# 1) strict clean pass
# 2) clean score terbesar
# 3) jika skor sama, lebih dekat centroid
candidates = candidates.sort_values(
    [
        "year",
        "strict_clean_pass",
        "clean_score",
        "centroid_distance_deg",
    ],
    ascending=[True, False, False, True],
).reset_index(drop=True)

candidates.to_csv(
    OUTPUT_DIR / "Figure7_all_clean_grid_candidates.csv",
    index=False,
)

summary_scan = (
    candidates.groupby("year", as_index=False)
    .agg(
        n_candidates=("clean_score", "size"),
        n_strict_pass=("strict_clean_pass", "sum"),
        best_score=("clean_score", "max"),
        median_score=("clean_score", "median"),
    )
)

print("\nCandidate scan summary:")
display(summary_scan)


## 5. Pilih kandidat terbaik per tahun

Jika ada kandidat `strict_clean_pass=True`, hanya kandidat strict tersebut yang dipertimbangkan untuk posisi #1. Jika tidak ada, skor tertinggi tetap dipilih dan `selection_mode` akan ditandai `fallback_best_score`.


In [ ]:
selected_rows = []
top_candidate_rows = []

for row in top_years.itertuples(index=False):
    year = int(row.year)

    sub = candidates.loc[candidates["year"] == year].copy()

    strict = sub.loc[sub["strict_clean_pass"]].copy()

    if len(strict):
        pool = strict.sort_values(
            ["clean_score", "centroid_distance_deg"],
            ascending=[False, True],
        )
        mode = "strict_clean"
    else:
        pool = sub.sort_values(
            ["clean_score", "centroid_distance_deg"],
            ascending=[False, True],
        )
        mode = "fallback_best_score"

    best = pool.iloc[0].copy()
    best["selection_mode"] = mode
    selected_rows.append(best)

    topn = sub.sort_values(
        [
            "strict_clean_pass",
            "clean_score",
            "centroid_distance_deg",
        ],
        ascending=[False, False, True],
    ).head(TOP_N_PREVIEW).copy()

    topn["rank_within_year"] = np.arange(1, len(topn) + 1)
    top_candidate_rows.append(topn)

selected = pd.DataFrame(selected_rows)
top_candidates = pd.concat(top_candidate_rows, ignore_index=True)

# Urutkan kembali sesuai ranking tahun nasional.
year_order = top_years["year"].astype(int).tolist()
selected["year_order"] = selected["year"].map(
    {y: i for i, y in enumerate(year_order)}
)
selected = selected.sort_values("year_order").drop(columns="year_order").reset_index(drop=True)

top_candidates["year_order"] = top_candidates["year"].map(
    {y: i for i, y in enumerate(year_order)}
)
top_candidates = (
    top_candidates
    .sort_values(["year_order", "rank_within_year"])
    .drop(columns="year_order")
    .reset_index(drop=True)
)

selected.to_csv(
    OUTPUT_DIR / "Figure7_selected_clean_representative_grids.csv",
    index=False,
)

top_candidates.to_csv(
    OUTPUT_DIR / "Figure7_top3_clean_candidates_per_year.csv",
    index=False,
)

cols_show = [
    "year",
    "national_peak_pentad",
    "latitude",
    "longitude",
    "clean_score",
    "strict_clean_pass",
    "selection_mode",
    "onset_time_pentads",
    "duration_pentads",
    "minimum_percentile",
    "recovery_value",
    "post_recovery_min",
    "post_recovery_mean",
    "immediate_relapse_below_p20",
    "second_event_start_post_recovery",
    "centroid_distance_deg",
]

print("Selected clean grids:")
display(selected[cols_show])


## 6. Plot helper

In [ ]:
def plot_candidate(ax, r, panel=None, show_info=True, preview=False):
    st = int(r["start_time_index"])
    crossing = int(r["crossing_time_index"])
    termination = int(r["termination_time_index"])
    gy = int(r["latitude_index"])
    gx = int(r["longitude_index"])

    left = max(0, st - PRE_ONSET_PENTADS)
    right = min(
        det.sizes["time_index"],
        termination + POST_RECOVERY_PENTADS + 1,
    )

    x = np.arange(left, right)

    y = np.asarray(
        pct_time
        .isel(
            time_index=slice(left, right),
            latitude=gy,
            longitude=gx,
        )
        .values,
        dtype=float,
    )

    # Full trajectory
    ax.plot(
        x,
        y,
        color="0.50",
        marker="o",
        markersize=3.0 if not preview else 2.3,
        linewidth=1.1,
        zorder=4,
    )

    # Event period
    ax.axvspan(
        st,
        termination,
        color="#fee8c8",
        alpha=0.38,
        zorder=0,
    )

    ax.fill_between(
        x,
        40,
        y,
        where=y < 40,
        color="#fcae91",
        alpha=0.38,
        zorder=1,
    )

    ax.fill_between(
        x,
        20,
        y,
        where=y < 20,
        color="#cb181d",
        alpha=0.36,
        zorder=2,
    )

    # Rapid onset
    xx = np.arange(st, crossing + 1)
    yy = np.asarray(
        pct_time
        .isel(
            time_index=slice(st, crossing + 1),
            latitude=gy,
            longitude=gx,
        )
        .values,
        dtype=float,
    )

    ax.plot(
        xx,
        yy,
        color="#b10026",
        marker="o",
        markersize=4,
        linewidth=2.2,
        zorder=6,
    )

    # Threshold
    ax.axhline(40, color="black", linestyle="--", linewidth=0.9)
    ax.axhline(20, color="black", linestyle=":", linewidth=1.0)

    y_st = float(
        pct_time.isel(
            time_index=st,
            latitude=gy,
            longitude=gx,
        ).values
    )
    y_cross = float(
        pct_time.isel(
            time_index=crossing,
            latitude=gy,
            longitude=gx,
        ).values
    )
    y_term = float(
        pct_time.isel(
            time_index=termination,
            latitude=gy,
            longitude=gx,
        ).values
    )

    ax.scatter(
        [st], [y_st],
        marker="s",
        color="#b2182b",
        s=42,
        zorder=8,
    )
    ax.scatter(
        [crossing], [y_cross],
        marker="v",
        color="#ef8a62",
        edgecolor="black",
        linewidth=0.25,
        s=50,
        zorder=8,
    )
    ax.scatter(
        [termination], [y_term],
        marker="^",
        color="#f46d43",
        edgecolor="black",
        linewidth=0.25,
        s=50,
        zorder=8,
    )

    ax.set_ylim(0, 100)
    ax.grid(alpha=0.13)

    ntick = min(6, len(x))
    tind = np.linspace(0, len(x) - 1, ntick, dtype=int)
    ax.set_xticks(x[tind])
    ax.set_xticklabels(
        [labels_full[left + i] for i in tind],
        rotation=30,
        ha="right",
        fontsize=7.5 if preview else 8,
    )

    title_prefix = f"({panel}) " if panel else ""
    ax.set_title(
        f"{title_prefix}{int(r['year'])}, national peak P{int(r['national_peak_pentad']):02d}\n"
        f"grid: {float(r['latitude']):.2f}°, {float(r['longitude']):.2f}°"
        + (
            f" | score={float(r['clean_score']):.1f}"
            if preview
            else ""
        ),
        fontsize=9.5 if preview else 10.5,
    )

    if show_info:
        info = (
            f"Onset = {int(r['onset_time_pentads'])} pentads\n"
            f"Speed = {float(r['onset_speed_pp_per_pentad']):.1f} pp pentad$^{{-1}}$\n"
            f"Episode = {int(r['duration_pentads'])} pentads\n"
            f"Severity = {float(r['severity_p40']):.0f} pp·pentad\n"
            f"Minimum = {float(r['minimum_percentile']):.1f}th pct.\n"
            f"Post-rec min = {float(r['post_recovery_min']):.1f}"
        )

        ax.text(
            0.015,
            0.97,
            info,
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=7.8 if preview else 8.3,
            bbox=dict(
                facecolor="white",
                edgecolor="0.65",
                alpha=0.9,
                boxstyle="round,pad=0.25",
            ),
            zorder=20,
        )


## 7. Preview tiga kandidat terbaik per tahun

In [ ]:
fig, axes = plt.subplots(
    4,
    TOP_N_PREVIEW,
    figsize=(16, 15),
    sharey=True,
)

for iyear, year in enumerate(year_order):
    sub = (
        top_candidates.loc[top_candidates["year"] == year]
        .sort_values("rank_within_year")
    )

    for j, (_, r) in enumerate(sub.iterrows()):
        ax = axes[iyear, j]
        plot_candidate(
            ax,
            r,
            panel=None,
            show_info=False,
            preview=True,
        )
        ax.set_title(
            f"{year} | candidate #{int(r['rank_within_year'])}\n"
            f"{float(r['latitude']):.2f}°, {float(r['longitude']):.2f}° | "
            f"score={float(r['clean_score']):.1f}\n"
            f"relapse={bool(r['immediate_relapse_below_p20'])}, "
            f"newFD={bool(r['second_event_start_post_recovery'])}",
            fontsize=9,
        )

        if j == 0:
            ax.set_ylabel("RZSM percentile")

fig.suptitle(
    "Top clean-lifecycle candidates within the largest drought-core cluster",
    fontsize=14,
    fontweight="bold",
)

fig.tight_layout(rect=[0, 0, 1, 0.97])
safe_savefig(fig, "Figure7_top3_candidate_preview.png")
plt.show()


## 8. Final Figure 7 — kandidat #1 per tahun

In [ ]:
fig, axes = plt.subplots(
    2,
    2,
    figsize=(15, 10),
    sharey=True,
)

letters = ["a", "b", "c", "d"]

for ax, (_, r), letter in zip(
    axes.ravel(),
    selected.iterrows(),
    letters,
):
    plot_candidate(
        ax,
        r,
        panel=letter,
        show_info=True,
        preview=False,
    )
    ax.set_ylabel("RZSM percentile")

axes[1, 0].set_xlabel("Pentad sequence")
axes[1, 1].set_xlabel("Pentad sequence")

legend_handles = [
    Patch(
        facecolor="#fee8c8",
        edgecolor="none",
        alpha=0.7,
        label="Detected drought episode",
    ),
    Patch(
        facecolor="#fcae91",
        edgecolor="none",
        alpha=0.7,
        label="P40 deficit",
    ),
    Patch(
        facecolor="#cb181d",
        edgecolor="none",
        alpha=0.55,
        label="P20 core deficit",
    ),
    Line2D(
        [0], [0],
        color="0.5",
        marker="o",
        lw=1,
        label="RZSM percentile",
    ),
    Line2D(
        [0], [0],
        color="#b10026",
        marker="o",
        lw=2,
        label="Rapid onset",
    ),
    Line2D(
        [0], [0],
        marker="s",
        color="none",
        markerfacecolor="#b2182b",
        label="Baseline",
    ),
    Line2D(
        [0], [0],
        marker="v",
        color="none",
        markerfacecolor="#ef8a62",
        label="First <P20",
    ),
    Line2D(
        [0], [0],
        marker="^",
        color="none",
        markerfacecolor="#f46d43",
        label="Recovery ≥P20",
    ),
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.005),
    ncol=4,
    frameon=False,
    fontsize=9,
)

fig.suptitle(
    "Figure 7. Representative clean-lifecycle grid evolution in the four strongest M2A years",
    fontsize=14.5,
    fontweight="bold",
)

fig.tight_layout(rect=[0, 0.085, 1, 0.95])

safe_savefig(
    fig,
    "Figure7_top4_clean_representative_events.png",
)

plt.show()


## 9. QC final

Yang perlu diperiksa:

- `strict_clean_pass=True` idealnya pada seluruh tahun;
- `immediate_relapse_below_p20=False`;
- `second_event_start_post_recovery=False`;
- `post_recovery_min >= 20`;
- preview kandidat #1 harus menunjukkan lifecycle yang masuk akal secara visual.

Jika salah satu tahun hanya memakai `fallback_best_score`, lihat preview top-3 dan pertimbangkan apakah `POST_RECOVERY_PENTADS` perlu diubah dari 4 menjadi 3 atau 5. Jangan mengubah tahun top-4; yang dievaluasi hanya grid di dalam episode utama tahun tersebut.


In [ ]:
qc_cols = [
    "year",
    "national_peak_pentad",
    "latitude",
    "longitude",
    "clean_score",
    "strict_clean_pass",
    "selection_mode",
    "onset_nonincrease_fraction",
    "pre_noncore_fraction",
    "core_le_p10_fraction",
    "core_std",
    "minimum_percentile",
    "recovery_value",
    "post_recovery_min",
    "post_recovery_mean",
    "immediate_relapse_below_p20",
    "second_event_start_post_recovery",
    "centroid_distance_deg",
]

display(selected[qc_cols])

if selected["immediate_relapse_below_p20"].any():
    print("WARNING: masih ada selected grid yang re-drop <P20 setelah recovery.")
else:
    print("PASS: tidak ada selected grid yang re-drop <P20 dalam post-recovery window.")

if selected["second_event_start_post_recovery"].any():
    print("WARNING: masih ada selected grid dengan event start baru segera setelah recovery.")
else:
    print("PASS: tidak ada event start baru segera setelah recovery.")

print("\nOutput folder:")
print(OUTPUT_DIR)

print("\nFiles:")
for name in [
    "Figure7_all_clean_grid_candidates.csv",
    "Figure7_top3_clean_candidates_per_year.csv",
    "Figure7_selected_clean_representative_grids.csv",
    "Figure7_top3_candidate_preview.png",
    "Figure7_top4_clean_representative_events.png",
]:
    print(" -", OUTPUT_DIR / name)
